# NurseGemma - Your Nursing AI Companion

**Built by a nurse, for nurses and families.**

NurseGemma bridges the gap between families and healthcare teams:

1. **Ask Questions** - Families get medical explanations they can understand
2. **Analyze Images** - Nurses get professional wound/scan documentation
3. **Nurse Summary** - Healthcare team sees what families asked and learned

*Families get answers. Nurses save time. Everyone stays informed.*

---

*Powered by Google MedGemma 1.5 | MedGemma Impact Challenge Submission*

In [ ]:
# Install dependencies
!pip install -q transformers accelerate pillow requests

In [ ]:
# Setup
import torch
import requests
from PIL import Image
from io import BytesIO
from datetime import datetime
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import login

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Initialize conversation log for nurse summary
conversation_log = []
print("Conversation tracking enabled for nurse summaries")

In [ ]:
# Authenticate with HuggingFace
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    login(token=secrets.get_secret('HF_TOKEN'))
    print("Authenticated via Kaggle Secrets")
except:
    login()  # Interactive login for Colab/local
    print("Authenticated interactively")

In [ ]:
# Load MedGemma 1.5 4B
MODEL_ID = "google/medgemma-1.5-4b-it"

print("Loading MedGemma 1.5...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
print(f"MedGemma loaded on {next(model.parameters()).device}")

---
## Feature 1: Ask Medical Questions

Get explanations that families can understand. No medical jargon.

In [ ]:
def ask_nursegemma(question: str, log_conversation: bool = True) -> str:
    """
    Ask a medical question and get a family-friendly explanation.
    Automatically logs the Q&A for the nurse summary.
    
    question: The question to ask
    log_conversation: If True, saves to conversation_log for nurse summary
    """
    prompt = f"""You are NurseGemma, a friendly nurse educator helping families understand medical information.

RULES:
- Use simple words (8th grade reading level)
- Avoid medical jargon - if you must use a term, explain it
- Be warm and reassuring
- Use helpful analogies when possible
- Keep answers concise but complete

QUESTION: {question}

ANSWER:"""
    
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=500, do_sample=False)
    
    response = processor.decode(output[0], skip_special_tokens=True)
    # Extract just the answer part
    if "ANSWER:" in response:
        response = response.split("ANSWER:")[-1].strip()
    
    # Log for nurse summary
    if log_conversation:
        conversation_log.append({
            "timestamp": datetime.now().strftime("%H:%M"),
            "type": "question",
            "question": question,
            "answer": response
        })
    
    return response

In [ ]:
# Example: Ask about a diagnosis
question = "What is CHF? My dad was just diagnosed and I'm scared."

print("Question:", question)
print("\n" + "="*50 + "\n")
print(ask_nursegemma(question))

In [ ]:
# Example: Ask about a medication
question = "Why does my mom take Lasix? What should we watch for?"

print("Question:", question)
print("\n" + "="*50 + "\n")
print(ask_nursegemma(question))

In [ ]:
# Example: Ask about a procedure
question = "My husband needs a cardiac catheterization. What happens during this test?"

print("Question:", question)
print("\n" + "="*50 + "\n")
print(ask_nursegemma(question))

---
## Feature 2: Medical Image Analysis

Upload wound photos or scans. Get professional nursing documentation.

In [ ]:
def analyze_wound(image_source, patient_context: str = "") -> str:
    """
    Analyze a wound image and generate nursing documentation.
    
    image_source: URL string or PIL Image
    patient_context: Optional context (e.g., "diabetic patient, sacral area")
    """
    # Load image
    if isinstance(image_source, str):
        response = requests.get(image_source, headers={"User-Agent": "NurseGemma"}, timeout=30)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = image_source
    
    context_line = f"\nPatient context: {patient_context}" if patient_context else ""
    
    prompt = f"""You are an experienced wound care nurse documenting a wound assessment.
{context_line}

Provide professional nursing documentation including:

1. WOUND TYPE & LOCATION
2. WOUND MEASUREMENTS (estimate from image)
3. WOUND BED CHARACTERISTICS
   - Tissue types (granulation, slough, eschar, epithelial)
   - Color and percentage
4. WOUND EDGES & PERIWOUND SKIN
5. DRAINAGE (if visible)
6. STAGING (for pressure injuries: Stage 1/2/3/4/Unstageable/DTPI)
7. RECOMMENDED INTERVENTIONS
8. FOLLOW-UP RECOMMENDATIONS

Use professional nursing terminology suitable for medical charting."""
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }]
    
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=800, do_sample=False)
    
    return processor.decode(output[0], skip_special_tokens=True)

In [ ]:
def analyze_scan(image_source, scan_type: str = "chest X-ray") -> str:
    """
    Analyze a medical scan and explain findings.
    
    image_source: URL string or PIL Image
    scan_type: Type of scan (e.g., "chest X-ray", "CT head", "MRI brain")
    """
    # Load image
    if isinstance(image_source, str):
        response = requests.get(image_source, headers={"User-Agent": "NurseGemma"}, timeout=30)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = image_source
    
    prompt = f"""You are an experienced nurse educator explaining a {scan_type} to help with clinical understanding.

Provide:

1. SCAN TYPE & QUALITY
2. KEY FINDINGS
   - Normal structures
   - Abnormalities (if any)
3. CLINICAL SIGNIFICANCE
   - What these findings might mean
4. NURSING IMPLICATIONS
   - What to monitor
   - When to notify the provider
5. PATIENT/FAMILY EXPLANATION
   - Simple explanation for the patient

Be thorough but clear."""
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }]
    
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=800, do_sample=False)
    
    return processor.decode(output[0], skip_special_tokens=True)

In [ ]:
# Example: Analyze a wound image
WOUND_URL = "https://upload.wikimedia.org/wikipedia/commons/f/fc/Grade3.jpg"

print("WOUND ASSESSMENT")
print("="*50)
print(analyze_wound(WOUND_URL, patient_context="elderly patient, sacral area"))

In [ ]:
# Example: Analyze a chest X-ray
CXR_URL = "https://upload.wikimedia.org/wikipedia/commons/8/87/X-ray_of_lobar_pneumonia.jpg"

print("CHEST X-RAY ANALYSIS")
print("="*50)
print(analyze_scan(CXR_URL, scan_type="chest X-ray"))

---
## Feature 3: Nurse Summary

Generate a summary for the healthcare team showing what families asked and learned. Saves nurses time and keeps everyone informed.

In [ ]:
def generate_nurse_summary(patient_name: str = "Patient") -> str:
    """
    Generate a summary for the nurse/healthcare team of all questions asked.
    This helps the nurse understand family concerns and knowledge gaps.
    
    patient_name: Optional patient identifier for the summary header
    """
    if not conversation_log:
        return "No conversations logged yet. Family hasn't asked any questions."
    
    # Build the questions list for the AI to summarize
    questions_text = "\n".join([
        f"- Q: {item['question']}\n  A: {item['answer'][:200]}..." 
        if len(item['answer']) > 200 else f"- Q: {item['question']}\n  A: {item['answer']}"
        for item in conversation_log if item['type'] == 'question'
    ])
    
    prompt = f"""You are NurseGemma creating a handoff summary for the bedside nurse.

The family has asked the following questions and received these answers:

{questions_text}

Create a brief NURSE HANDOFF SUMMARY that includes:

1. KEY CONCERNS: What is the family most worried about? (2-3 bullet points)
2. TOPICS COVERED: What medical topics were explained? (brief list)
3. KNOWLEDGE GAPS: What might need further clarification from the nurse?
4. EMOTIONAL STATE: Based on questions, how does the family seem to be coping?
5. SUGGESTED FOLLOW-UP: What should the nurse address or reinforce?

Keep it concise - nurses are busy. Use bullet points."""
    
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=600, do_sample=False)
    
    summary = processor.decode(output[0], skip_special_tokens=True)
    
    # Format the output
    header = f"""
{'='*60}
NURSEGEMMA HANDOFF SUMMARY
Patient: {patient_name}
Time: {datetime.now().strftime("%Y-%m-%d %H:%M")}
Questions Asked: {len([x for x in conversation_log if x['type'] == 'question'])}
{'='*60}
"""
    return header + summary


def clear_conversation_log():
    """Clear the conversation log (e.g., for a new patient/shift)."""
    global conversation_log
    conversation_log = []
    print("Conversation log cleared. Ready for new patient.")

In [ ]:
# Example: Generate a nurse summary after family questions
# (The questions from earlier cells are automatically logged)

print(generate_nurse_summary(patient_name="Room 512 - Mr. Johnson"))

---
## Interactive Mode

Use these functions with your own questions and images!

In [ ]:
# YOUR TURN: Ask your own question
my_question = "What does it mean when they say my blood pressure is high?"

print("Your Question:", my_question)
print("\n" + "="*50 + "\n")
print(ask_nursegemma(my_question))

In [ ]:
# YOUR TURN: Analyze your own image
# Replace with your own image URL or upload an image
my_image_url = "YOUR_IMAGE_URL_HERE"

# Uncomment the type of analysis you need:
# print(analyze_wound(my_image_url, patient_context="describe patient"))
# print(analyze_scan(my_image_url, scan_type="chest X-ray"))

---

## About NurseGemma

As an ICU nurse, I spend 40% of my shift documenting instead of caring for patients. Meanwhile, families wait anxiously with questions - "What does this mean?" "Why is this beeping?" "Is my dad going to be okay?"

**NurseGemma bridges this gap:**

1. **Family at bedside** → Asks NurseGemma questions while waiting
2. **NurseGemma responds** → Provides clear, reassuring explanations
3. **Nurse makes rounds** → Reviews the summary of what family asked
4. **Nurse follows up** → Clarifies, expands, or corrects as needed

*Families get immediate answers. Nurses save time on repetitive education. Everyone stays on the same page.*

**Disclaimer**: NurseGemma is an educational tool. All outputs should be verified by qualified healthcare professionals. Not for diagnostic or treatment decisions.

---

*Built for the MedGemma Impact Challenge 2026*